# Part 1 - Python and Data Foundations
**Care Look Clinics No-Show Guard** - ingest, clean, engineer features, enrich with a live API, and export for SQL.

**How to run:** run the cells from top to bottom (Runtime > Run all in Colab).
Input files are read from the `data/` folder. Output files are written next to this notebook.

In [1]:
# ---- Setup (works on your laptop AND in Google Colab) ----
import os, subprocess

# Only needed in Colab: if the data folder is not here yet, clone your GitHub repo.
# Replace the URL with YOUR repository URL after you upload the project to GitHub.
REPO_URL = "https://github.com/abhiipanditt/care-look-clinic-no-show-guard.git"

if not os.path.exists("data/appointments.csv"):
    subprocess.run(["git", "clone", REPO_URL, "project"], check=True)
    os.chdir("project")

print("Working folder:", os.getcwd())
print("Files here:", os.listdir("."))

Working folder: /content/project
Files here: ['04_genai.ipynb', '01_ingest_clean.ipynb', '03_model.ipynb', 'clean_appointments.csv', 'data', 'test_risk_score.csv', '.git', 'all_risk_scores.csv', 'rejected_rows.json', 'load_data.sql', '02_queries.sql', 'README.md']


## Task 1.1 - File ingestion with the `csv` module (no Pandas here)
`DictReader` reads each row as a dictionary. `parse_rows()` converts types and
**rejects** any row that cannot be converted (for example a blank age or blank distance).

In [2]:
import csv, json

RAW_PATH = "data/appointments.csv"

with open(RAW_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    headers = reader.fieldnames
    raw_rows = list(reader)

print("Headers:", headers)
print("Row count:", len(raw_rows))


def parse_rows(rows):
    """Convert types row by row. Returns (accepted_rows, rejected_rows)."""
    accepted, rejected = [], []
    for line_no, row in enumerate(rows, start=2):  # line 1 of the file is the header
        try:
            parsed = dict(row)
            parsed["age"] = int(row["age"])                      # int
            parsed["fee_inr"] = int(row["fee_inr"])              # int
            parsed["prior_no_shows"] = int(row["prior_no_shows"])  # int
            parsed["distance_km"] = float(row["distance_km"])    # float
            accepted.append(parsed)
        except (ValueError, TypeError, KeyError) as err:
            rejected.append({"csv_line": line_no, "reason": str(err), "row": row})
    return accepted, rejected


accepted_rows, rejected_rows = parse_rows(raw_rows)
print("Accepted rows:", len(accepted_rows))   # expected 988
print("Rejected rows:", len(rejected_rows))   # expected 32

with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected_rows, f, indent=2, ensure_ascii=False)
print("Saved rejected_rows.json")

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020
Accepted rows: 988
Rejected rows: 32
Saved rejected_rows.json


**What got rejected?** A quick look at the reasons (blank age / blank distance).

In [3]:
from collections import Counter
print(Counter(r["reason"] for r in rejected_rows))

Counter({"could not convert string to float: ''": 20, "invalid literal for int() with base 10: ''": 12})


> Note: Task 1.1 shows how strict type-conversion catches bad rows. From Task 1.2 onward we re-load the
> **full file with Pandas** and *fix* missing ages/distances by imputation (Task 1.3), as the brief asks.

## Task 1.2 - Inspection, cleaning and parsing (Pandas)

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(RAW_PATH)
print("Rows after read_csv:", len(df))
print(df.dtypes)
print("\nMissing values per column:\n", df.isna().sum())

Rows after read_csv: 1020
appointment_id      object
patient_id          object
clinic_id           object
booked_on           object
appt_date           object
age                float64
gender              object
distance_km        float64
fee_inr              int64
booking_channel     object
visit_type          object
reminder_sent       object
prior_no_shows       int64
status              object
dtype: object

Missing values per column:
 appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64


In [5]:
# Step 1: drop exact duplicate rows
n_dups = df.duplicated().sum()
print("Exact duplicate rows found:", n_dups)   # expected 20
df = df.drop_duplicates().reset_index(drop=True)
print("Rows after dropping duplicates:", len(df))

Exact duplicate rows found: 20
Rows after dropping duplicates: 1000


In [6]:
# Step 2: clean booking_channel (strip spaces, Title Case)
before = df["booking_channel"].copy()
print("Values BEFORE cleaning:", sorted(before.unique()))
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Values AFTER cleaning :", sorted(df["booking_channel"].unique()))
print("Rows whose booking_channel text was changed:", (before != df["booking_channel"]).sum())
print("Rows after channel cleaning:", len(df))

Values BEFORE cleaning: [' App', ' phone', ' walk-in', 'APP ', 'App', 'App ', 'PHONE', 'Phone', 'Phone ', 'WALK-IN', 'Walk-In', 'Walk-in', 'Walk-in ', 'app', 'phone', 'walk-in']
Values AFTER cleaning : ['App', 'Phone', 'Walk-In']
Rows whose booking_channel text was changed: 295
Rows after channel cleaning: 1000


In [7]:
# Step 3: convert dates. In THIS file dates look like 2025-01-25 (year-month-day), so the format is %Y-%m-%d.
# errors="coerce" turns impossible values ("TBD", "N/A", "not booked", 2025-06-31) into NaT (missing date).
DATE_FORMAT = "%Y-%m-%d"
print("Non-date text found in appt_date:",
      df.loc[~df["appt_date"].fillna("").str.match(r"^\d{4}-\d{2}-\d{2}$"), "appt_date"].tolist())

df["appt_date"] = pd.to_datetime(df["appt_date"], format=DATE_FORMAT, errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"], format=DATE_FORMAT, errors="coerce")
print("Missing appt_date after conversion:", df["appt_date"].isna().sum())
print("Rows after date conversion:", len(df))

Non-date text found in appt_date: ['not booked', 'TBD', nan]
Missing appt_date after conversion: 6
Rows after date conversion: 1000


In [8]:
# Step 4: drop rows with a missing date or a missing status
df = df.dropna(subset=["appt_date", "status"]).reset_index(drop=True)
print("Rows after dropping missing date/status:", len(df))   # expected 990
print(df["status"].value_counts())

Rows after dropping missing date/status: 990
status
Attended    772
No-Show     218
Name: count, dtype: int64


## Task 1.3 - Validity, gaps and outliers

In [9]:
# Validity: an age below 1 or above 100 is impossible -> set to missing
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages found:", invalid_age_mask.sum(), "->", sorted(df.loc[invalid_age_mask, "age"].tolist()))  # expected 5
df.loc[invalid_age_mask, "age"] = np.nan

# Gaps: impute with the median
missing_age = df["age"].isna().sum()
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median)
print(f"Missing ages imputed: {missing_age} with median = {age_median}")   # expected 17, median 48

missing_dist = df["distance_km"].isna().sum()
dist_median = df["distance_km"].median()
df["distance_km"] = df["distance_km"].fillna(dist_median)
print(f"Missing distances imputed: {missing_dist} with median = {dist_median}")   # median 8.25

Invalid ages found: 5 -> [-4.0, 0.0, 150.0, 199.0, 250.0]
Missing ages imputed: 17 with median = 48.0
Missing distances imputed: 20 with median = 8.25


In [10]:
# Outliers: IQR rule, cap values at Q3 + 1.5 * IQR
def cap_upper_iqr(series, name):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    n_capped = (series > upper).sum()
    print(f"{name}: Q1={q1}, Q3={q3}, IQR={iqr}, upper fence={upper}, values capped={n_capped}")
    return series.clip(upper=upper)

df["distance_km"] = cap_upper_iqr(df["distance_km"], "distance_km")
df["fee_inr"] = cap_upper_iqr(df["fee_inr"], "fee_inr")

# Round: distance to 1 decimal, fee to whole rupees
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round(0).astype(int)
df["age"] = df["age"].astype(int)
print(df[["age", "distance_km", "fee_inr"]].describe())

distance_km: Q1=4.625, Q3=13.0, IQR=8.375, upper fence=25.5625, values capped=26
fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, upper fence=1218.75, values capped=3
              age  distance_km      fee_inr
count  990.000000   990.000000   990.000000
mean    48.173737     9.500909   635.714141
std     16.655376     6.160314   183.802489
min     18.000000     0.600000   300.000000
25%     34.000000     4.625000   500.000000
50%     48.000000     8.200000   650.000000
75%     62.000000    13.000000   787.500000
max     78.000000    25.600000  1219.000000


## Task 1.4 - Feature engineering and NumPy analysis

In [11]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())   # expected 1 to 45
print(f"No-shows: {df['no_show'].sum()} out of {len(df)} rows")                 # expected 218 of 990

# Feature matrix as a NumPy array
feature_cols = ["age", "distance_km", "fee_inr", "lead_days"]
X = df[feature_cols].to_numpy(dtype=float)
print("X shape:", X.shape)

# Mean of each column
for name, m in zip(feature_cols, X.mean(axis=0)):
    print(f"Mean {name}: {m:.2f}")

# lead_days > 14
lead = X[:, 3]
n_long = int((lead > 14).sum())
print(f"Rows with lead_days > 14: {n_long} ({n_long / len(lead):.1%})")

# Z-scores for distance_km, and how many rows have |z| < 2
dist = X[:, 1]
z = (dist - dist.mean()) / dist.std()
print("Rows with |z-score| < 2 for distance_km:", int((np.abs(z) < 2).sum()))

lead_days range: 1 to 45
No-shows: 218 out of 990 rows
X shape: (990, 4)
Mean age: 48.17
Mean distance_km: 9.50
Mean fee_inr: 635.71
Mean lead_days: 11.82
Rows with lead_days > 14: 296 (29.9%)
Rows with |z-score| < 2 for distance_km: 937


## Task 1.5 - Enrich the clinics file with a live API (Open-Meteo, free, no key needed)
For each of the 8 clinics we ask for the current temperature using its latitude/longitude.
Timeouts are handled with a retry loop. **This cell needs internet** (Colab has it).

In [12]:
import time
import requests

clinics = pd.read_csv("data/clinics.csv")
print(clinics)

API_URL = "https://api.open-meteo.com/v1/forecast"


def get_temperature(lat, lon, retries=3, timeout=10):
    """Return current temperature (deg C) for a location, or None if all retries fail."""
    params = {"latitude": lat, "longitude": lon, "current": "temperature_2m"}
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(API_URL, params=params, timeout=timeout)
            resp.raise_for_status()
            return resp.json()["current"]["temperature_2m"]
        except (requests.exceptions.RequestException, KeyError, ValueError) as err:
            print(f"  attempt {attempt}/{retries} failed: {err}")
            time.sleep(2 * attempt)   # wait a little longer each time
    return None


temps = []
for _, row in clinics.iterrows():
    t = get_temperature(row["latitude"], row["longitude"])
    print(f"{row['clinic_id']} {row['city']}: {t}")
    temps.append(t)

clinics["temperature_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print("Saved clinics_enriched.csv")
clinics

  clinic_id               clinic_name     city    tier  latitude  longitude
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462
C01 Pune: 29.1
C02 Pune: 29.9
C03 Nagpur: 26.9
C04 Jaipur: 24.2
C05 Jaipur: 24.2
C06 Kochi: 31.2
C07 Indore: 26.7
C08 Lucknow: 26.2
Saved clinics_enriched.csv


,clinic_id,clinic_name,city,tier,latitude,longitude,temperature_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,29.1
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,29.9
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,26.9
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,24.2
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,24.2
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,31.2
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,26.7
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,26.2


## Task 1.6 - Export clean data and generate SQL INSERT statements

In [13]:
export_cols = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
               "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent",
               "prior_no_shows", "status", "lead_days", "no_show"]

clean = df.sort_values("appointment_id").reset_index(drop=True)[export_cols].copy()
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")

# Safety checks that match the acceptance criteria
assert len(clean) == 990, len(clean)
assert clean.shape[1] == 16, clean.shape[1]
assert clean.isna().sum().sum() == 0, "there are still missing values"
clean.to_csv("clean_appointments.csv", index=False)
print("Saved clean_appointments.csv ->", clean.shape)
clean.head()

Saved clean_appointments.csv -> (990, 16)


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [14]:
def sql_value(v):
    """Turn a Python value into a safe SQL literal."""
    if pd.isna(v):
        return "NULL"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        return repr(float(v))
    return "'" + str(v).replace("'", "''") + "'"   # escape single quotes


def make_inserts(frame, table):
    cols = ", ".join(frame.columns)
    lines = []
    for row in frame.itertuples(index=False):
        values = ", ".join(sql_value(v) for v in row)
        lines.append(f"INSERT INTO {table} ({cols}) VALUES ({values});")
    return lines


clinics_base = pd.read_csv("data/clinics.csv")   # original 6 columns (not the temperature one)

with open("load_data.sql", "w", encoding="utf-8") as f:
    f.write("-- Care Look Clinics No-Show Guard: data load script\n")
    f.write("-- Clinics FIRST (parent table), appointments AFTER (they reference clinic_id).\n\n")
    f.write("\n".join(make_inserts(clinics_base, "clinics")) + "\n\n")
    f.write("\n".join(make_inserts(clean, "appointments")) + "\n")

n_lines = sum(1 for _ in open("load_data.sql", encoding="utf-8"))
print("Saved load_data.sql with", n_lines, "lines (8 clinics + 990 appointments + comments/blank lines)")

Saved load_data.sql with 1002 lines (8 clinics + 990 appointments + comments/blank lines)
